In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load the Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

# 2. Define Target, Leaks, and ID Column
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'

# These are the features causing the 0.9+ R2 data leakage. 
# We MUST drop them from both the training and testing sets.
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

# Prepare Training Data
y_train = train_df[TARGET]
X_train = train_df.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

# Prepare Testing Data (Save IDs for the final submission format)
test_ids = test_df[ID_COL]
X_test = test_df.drop(columns=[ID_COL] + LEAKED_FEATURES)

# 3. Build a Preprocessing Pipeline
# Identify which columns are text/categorical and which are numbers
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Fill missing numbers with the median
numeric_transformer = SimpleImputer(strategy='median')

# Fill missing text with the most frequent value, then convert to numbers.
# handle_unknown='use_encoded_value' ensures the model doesn't crash if 
# the test set contains a category it never saw during training.
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 4. Define and Train the Model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training model (this might take a moment)...")
model.fit(X_train, y_train)

# 5. Predict on Test Data and Format Submission
print("Predicting on test data...")
predictions = model.predict(X_test)

# Create the final dataframe matching the sample_submission format
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: predictions
})

# Save to CSV
submission.to_csv('submissionDay9.csv', index=False)
print("Saved predictions to 'submission.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_31412\3626102385.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training model (this might take a moment)...
Predicting on test data...
Saved predictions to 'submission.csv'


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Build the Feature Engineering Function
def engineer_features(df):
    data = df.copy()
    
    # Feature 1: Speed Advantage (Who moves first)
    data['speed_advantage'] = data['speed_stat_pikachu'] - data['speed_stat_opponent']
    
    # Feature 2: Level Ratio (Relative Power)
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5) # Added a tiny number to prevent divide by zero
    
    # Feature 3: Attack/Defense Matchup Ratio
    # If the move is Special, compare sp_attack to sp_defense. 
    # Otherwise, compare regular attack to defense.
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['attack_stat'] / (data['defense_stat'] + 1e-5)
    )
    
    # Feature 4: Theoretical Power of the Turn
    # Combine move power, effectiveness, and stat ratios into one number
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # Feature 5: Previous HP State
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)

    danger = np.ones(len(data))
    
    # If the weather matches the opponent's type, their attacks will hit Pikachu much harder
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    
    # Hail and Sandstorm deal chip damage every turn to Pikachu, lowering expected HP
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    
    # If Electric Terrain is active, Pikachu gets a power boost, lowering the danger
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    
    data['weather_danger_level'] = danger
        
    return data

# Apply the new features to both datasets
print("Engineering features...")
train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Define Model and Train
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training model with new features...")
model.fit(X_train, y_train)

# 6. Predict and Create Submission
print("Predicting on test data...")
predictions = model.predict(X_test)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: predictions
})

submission.to_csv('submissionDay9.csv', index=False)
print("Saved predictions to 'engineered_submission.csv'")

Engineering features...
Training model with new features...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_31412\1263305439.py:77: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Predicting on test data...
Saved predictions to 'engineered_submission.csv'


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering Function
def engineer_features(df):
    data = df.copy()
    
    # 2.1 Stat Differentials
    data['speed_advantage'] = data['speed_stat_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    # 2.2 Attack/Defense Ratios based on Physical vs Special moves
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['attack_stat'] / (data['defense_stat'] + 1e-5)
    )
    
    # 2.3 Theoretical Power combination
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # 2.4 Relative HP state
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    # 2.5 Weather Danger Level Mapping
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger


    data['adj_speed_pikachu'] = np.where(data['pikachu_status'] == 'Paralyzed', 
                                         data['speed_stat_pikachu'] * 0.5, 
                                         data['speed_stat_pikachu'])
    
    data['adj_attack_pikachu'] = np.where(data['pikachu_status'] == 'Burned', 
                                          data['attack_stat'] * 0.5, 
                                          data['attack_stat'])

    # 1. Stat Differentials (Using Adjusted Speed)
    data['speed_advantage'] = data['adj_speed_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    # 2. Attack/Defense Ratios (Using Adjusted Attack)
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack_pikachu'] / (data['defense_stat'] + 1e-5)
    )
    
    # --- NEW: STAB (Same Type Attack Bonus) ---
    # Pikachu is Electric. If the move is Electric, it hits 1.5x harder.
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    
    # 3. Theoretical Power (Now including STAB)
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['stab_multiplier'] * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # 4. Relative HP state
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    # 5. Weather Danger Level Mapping
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger

    return data

# Apply feature engineering
print("Engineering features...")
train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

# FILL MISSING MAX_HP to prevent NaNs during the clipping step
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Define Gradient Boosting Model
hgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        max_iter=300, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42
    ))
])

# Train
print("Training Gradient Boosting model...")
hgb_model.fit(X_train, y_train)

# 6. Predict and Post-Process (Clipping)
print("Predicting and applying physics constraints...")
raw_predictions = hgb_model.predict(X_test)

# Force predictions to be physically possible (0 to max_hp)
clipped_predictions = np.clip(raw_predictions, 0, test_max_hp)

# Create submission
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

submission.to_csv('submissionDay10.csv', index=False)
print("Success! Saved clean predictions to 'SubmissionDay10.csv'")

Engineering features...
Training Gradient Boosting model...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_25748\1336641661.py:113: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Predicting and applying physics constraints...
Success! Saved clean predictions to 'SubmissionDay10.csv'


In [ ]:
# import pandas as pd  used to combination of random trees and xgboost to avg out the errors and find out the correct output but did not work that well
# import numpy as np   so now increased the weight of xgboost but that backfired as well so xgboost is not the correct model iguess so switched to the 
# import xgboost as xgb  og histboost 
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OrdinalEncoder

# # 1. Load Data
# train_df = pd.read_csv('train.csv')
# test_df = pd.read_csv('test.csv')

# # 2. Feature Engineering
# def engineer_features(df):
#     data = df.copy()
#     data['speed_advantage'] = data['speed_stat_pikachu'] - data['speed_stat_opponent']
#     data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
#     data['attack_defense_ratio'] = np.where(
#         data['move_category'] == 'Special',
#         data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
#         data['attack_stat'] / (data['defense_stat'] + 1e-5)
#     )
    
#     data['theoretical_power'] = (
#         data['move_power'].fillna(0) * 
#         data['type_effectiveness'].fillna(1.0) * 
#         data['attack_defense_ratio'].fillna(1.0)
#     )
    
#     if 'previous_hp' in data.columns and 'max_hp' in data.columns:
#         data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
#     danger = np.ones(len(data))
#     danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
#     danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
#     danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
#     danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
#     data['weather_danger_level'] = danger
    
#     return data

# print("Engineering features...")
# train_feat = engineer_features(train_df)
# test_feat = engineer_features(test_df)

# # 3. Setup Target, Remove Leaks, and Fill Missing max_hp
# TARGET = 'pikachu_hp'
# ID_COL = 'battle_turn'
# LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

# y_train = train_feat[TARGET]
# X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

# test_ids = test_feat[ID_COL]
# X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)
# test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# # 4. Build Preprocessing Pipeline
# categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
# numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# numeric_transformer = SimpleImputer(strategy='median')
# categorical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', numeric_transformer, numeric_cols),
#         ('cat', categorical_transformer, categorical_cols)
#     ])

# # 5. Define BOTH Models
# print("Initializing models...")
# rf_model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor(
#         n_estimators=150, 
#         max_depth=12, 
#         random_state=42, 
#         n_jobs=-1
#     ))
# ])

# xgb_model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', xgb.XGBRegressor(
#         n_estimators=300, 
#         learning_rate=0.05, 
#         max_depth=6, 
#         subsample=0.8,
#         colsample_bytree=0.8,
#         random_state=42,
#         n_jobs=-1
#     ))
# ])

# # 6. Train Models
# print("Training Random Forest...")
# rf_model.fit(X_train, y_train)

# print("Training XGBoost...")
# xgb_model.fit(X_train, y_train)

# # 7. Predict and Average (The Ensemble)
# print("Generating predictions...")
# rf_preds = rf_model.predict(X_test)
# xgb_preds = xgb_model.predict(X_test)

# # Give XGBoost 85% weight, and Random Forest 15% weight
# ensemble_preds = (0.85 * xgb_preds) + (0.15 * rf_preds)

# # Post-processing: Force physical boundaries
# clipped_preds = np.clip(ensemble_preds, 0, test_max_hp)

# # 8. Create Submission
# submission = pd.DataFrame({
#     ID_COL: test_ids,
#     TARGET: clipped_preds
# })

# submission.to_csv('submissionDay10.csv', index=False)
# print("Success! Saved clean predictions to 'submissionDay10.csv'")

Engineering features...
Initializing models...
Training Random Forest...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_25748\2312905063.py:61: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training XGBoost...
Generating predictions...
Success! Saved clean predictions to 'ensemble_xgb_rf_submission.csv'


In [ ]:
# import pandas as pd  adopted a new ouput mechanism instead of calculating the final hp we are finding the damage dealt and it backfired
# import numpy as np 
# from sklearn.ensemble import HistGradientBoostingRegressor
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OrdinalEncoder

# # 1. Load Data
# train_df = pd.read_csv('train.csv')
# test_df = pd.read_csv('test.csv')

# # 2. Feature Engineering (Keeping all your powerful mechanics!)
# def engineer_features(df):
#     data = df.copy()
    
#     data['adj_speed_pikachu'] = np.where(data['pikachu_status'] == 'Paralyzed', data['speed_stat_pikachu'] * 0.5, data['speed_stat_pikachu'])
#     data['adj_attack_pikachu'] = np.where(data['pikachu_status'] == 'Burned', data['attack_stat'] * 0.5, data['attack_stat'])

#     data['speed_advantage'] = data['adj_speed_pikachu'] - data['speed_stat_opponent']
#     data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
#     data['attack_defense_ratio'] = np.where(
#         data['move_category'] == 'Special',
#         data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
#         data['adj_attack_pikachu'] / (data['defense_stat'] + 1e-5)
#     )
    
#     data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    
#     data['theoretical_power'] = (
#         data['move_power'].fillna(0) * 
#         data['type_effectiveness'].fillna(1.0) * 
#         data['stab_multiplier'] * 
#         data['attack_defense_ratio'].fillna(1.0)
#     )
    
#     if 'previous_hp' in data.columns and 'max_hp' in data.columns:
#         data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
#     danger = np.ones(len(data))
#     danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
#     danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
#     danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
#     danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
#     data['weather_danger_level'] = danger
    
#     return data

# train_feat = engineer_features(train_df)
# test_feat = engineer_features(test_df)

# # 3. TARGET TRANSFORMATION (The Game Changer)
# # Drop rows in training where previous_hp is missing so our math doesn't break
# train_feat = train_feat.dropna(subset=['previous_hp', 'pikachu_hp'])

# # Create the new target: How much did the HP change this turn?
# train_feat['delta_hp'] = train_feat['pikachu_hp'] - train_feat['previous_hp']
# TARGET = 'delta_hp' 
# ID_COL = 'battle_turn'

# # These are the actual leaks present in both datasets
# LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied'] 

# y_train = train_feat[TARGET]

# # Drop ID, Target, Leaks, AND 'pikachu_hp' from the training set
# X_train = train_feat.drop(columns=[ID_COL, TARGET, 'pikachu_hp'] + LEAKED_FEATURES)

# # Drop ID and Leaks from the test set (using errors='ignore' just to be safe!)
# X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')

# # Impute missing max_hp and previous_hp in the test set to allow final math reconstruction
# test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())
# test_previous_hp = test_feat['previous_hp'].fillna(test_max_hp)

# # 4. Pipeline Setup
# categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
# numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# numeric_transformer = SimpleImputer(strategy='median')
# categorical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', numeric_transformer, numeric_cols),
#         ('cat', categorical_transformer, categorical_cols)
#     ])

# # 5. Train the Model on the Delta
# hgb_model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', HistGradientBoostingRegressor(
#         max_iter=400,          # Boosted iterations slightly
#         learning_rate=0.04,    # Slowed down learning rate for precision
#         max_depth=7,           
#         random_state=42
#     ))
# ])

# print("Training model to predict damage dealt...")
# hgb_model.fit(X_train, y_train)

# # 6. Predict and Reconstruct Absolute HP
# print("Reconstructing final HP...")
# predicted_delta = hgb_model.predict(X_test)

# # Final HP = Previous HP + Predicted Change
# reconstructed_hp = test_previous_hp + predicted_delta

# # Apply the laws of physics (HP can't be less than 0 or greater than Max HP)
# final_predictions = np.clip(reconstructed_hp, 0, test_max_hp)

# submission = pd.DataFrame({
#     ID_COL: test_feat[ID_COL],
#     'pikachu_hp': final_predictions
# })

# submission.to_csv('submissionDay10.csv', index=False)
# print("Saved predictions to 'submissionDay10.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_25748\1607719605.py:78: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training model to predict damage dealt...
Reconstructing final HP...
Saved predictions to 'submissionDay10.csv'


In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering Function (Cleaned up duplicates, kept ALL features)
def engineer_features(df):
    data = df.copy()
    
    # 2.1 Status Condition Adjustments
    data['adj_speed_pikachu'] = np.where(data['pikachu_status'] == 'Paralyzed', 
                                         data['speed_stat_pikachu'] * 0.5, 
                                         data['speed_stat_pikachu'])
    
    data['adj_attack_pikachu'] = np.where(data['pikachu_status'] == 'Burned', 
                                          data['attack_stat'] * 0.5, 
                                          data['attack_stat'])

    # 2.2 Stat Differentials (Using Adjusted Speed)
    data['speed_advantage'] = data['adj_speed_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    # 2.3 Attack/Defense Ratios (Using Adjusted Attack)
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack_pikachu'] / (data['defense_stat'] + 1e-5)
    )
    
    # 2.4 STAB (Same Type Attack Bonus)
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    
    # 2.5 Theoretical Power
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['stab_multiplier'] * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # 2.6 Relative HP state
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    # 2.7 Weather Danger Level Mapping
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger

    return data

# Apply feature engineering
print("Engineering features...")
train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

# FILL MISSING MAX_HP to prevent NaNs during the clipping step
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Define Gradient Boosting Model (Base Pipeline)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

# 6. The Tuning Grid (The Brute-Force Engine)
param_distributions = {
    'regressor__learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'regressor__max_iter': [300, 400, 500, 600],  # Tests more trees
    'regressor__max_depth': [5, 6, 7, 9, 12, None], 
    'regressor__max_leaf_nodes': [31, 50, 75, 120],
    'regressor__l2_regularization': [0.0, 0.1, 0.5, 1.0, 5.0]
}

print("Running Randomized Search (This will take a few minutes)...")
# Tests 30 random combinations of the parameters above, cross-validating 3 times each (90 total fits)
search = RandomizedSearchCV(
    pipeline, 
    param_distributions, 
    n_iter=30, 
    cv=3, 
    scoring='r2', 
    random_state=42, 
    n_jobs=-1
)

search.fit(X_train, y_train)

print(f"Best internal R2 score: {search.best_score_}")
print(f"Best parameters found:\n {search.best_params_}")

# 7. Predict and Post-Process using the absolute Best Model found
print("Predicting and applying physics constraints with the Champion Model...")
best_model = search.best_estimator_
raw_predictions = best_model.predict(X_test)

# Force predictions to be physically possible (0 to max_hp)
clipped_predictions = np.clip(raw_predictions, 0, test_max_hp)

# Create submission
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

submission.to_csv('SubmissionDay10.csv', index=False)
print("Success! Saved clean predictions to 'SubmissionDay10_Tuned.csv'")

Engineering features...
Running Randomized Search (This will take a few minutes)...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_25748\324122292.py:83: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Best internal R2 score: 0.49555463106509573
Best parameters found:
 {'regressor__max_leaf_nodes': 75, 'regressor__max_iter': 500, 'regressor__max_depth': 9, 'regressor__learning_rate': 0.01, 'regressor__l2_regularization': 5.0}
Predicting and applying physics constraints with the Champion Model...
Success! Saved clean predictions to 'SubmissionDay10_Tuned.csv'


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering (Your exact winning setup)
def engineer_features(df):
    data = df.copy()
    data['adj_speed_pikachu'] = np.where(data['pikachu_status'] == 'Paralyzed', data['speed_stat_pikachu'] * 0.5, data['speed_stat_pikachu'])
    data['adj_attack_pikachu'] = np.where(data['pikachu_status'] == 'Burned', data['attack_stat'] * 0.5, data['attack_stat'])

    data['speed_advantage'] = data['adj_speed_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack_pikachu'] / (data['defense_stat'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['stab_multiplier'] * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
# We still encode to integers so the model can read them, but we will tell the model they are categories!
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# --- NEW: Create a boolean mask to tell the model which columns are categories ---
# The ColumnTransformer outputs numeric columns first, then categorical columns
categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

# 5. Define Gradient Boosting Model (Unlocking Native Categorical Support)
hgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        categorical_features=categorical_mask, # <--- THIS IS THE MAGIC FIX
        max_iter=350, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42
    ))
])

# Train
print("Training Gradient Boosting model with Native Categorical Splitting...")
hgb_model.fit(X_train, y_train)

# 6. Predict and Post-Process (Clipping)
print("Predicting and applying physics constraints...")
raw_predictions = hgb_model.predict(X_test)
clipped_predictions = np.clip(raw_predictions, 0, test_max_hp)

# Create submission
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

submission.to_csv('submissionDay11.csv', index=False)
print("Success! Saved clean predictions to 'submissionDay11'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_12568\2043872396.py:81: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training Gradient Boosting model with Native Categorical Splitting...
Predicting and applying physics constraints...
Success! Saved clean predictions to 'submissionDay11'
